# DeBERTA + LoRA fine tuning

## 1. Setup & Imports

In [ ]:
!pip install -q peft
!pip uninstall torchao -y

import os, re, random, warnings
os.environ["WANDB_MODE"] = "disabled"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
from collections import Counter
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A','B','C','D','E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'


## 2. Data Loading & EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

plt.figure(figsize=(6,4))
train_df['answer'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Answer Distribution (Train)')
plt.xlabel('Option'); plt.ylabel('Count')
plt.show()

def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)
train_df['option_set'] = train_df.apply(option_set_key, axis=1)
test_df['option_set'] = test_df.apply(option_set_key, axis=1)

dup_count = train_df.duplicated(subset=['option_set']).sum()
print(f"Duplicate option-sets in train: {dup_count}")

train_option_sets = set(train_df['option_set'].unique())
test_matched = test_df['option_set'].isin(train_option_sets)
print(f"Test questions matched to train: {test_matched.sum()} / {len(test_df)} ({test_matched.mean()*100:.1f}%)")

## 3. Data Preprocessing & Strict Splitting

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d: uf.union(i, d[k])
        else: d[k] = i
train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))
train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l:i for i,l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)
y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero group overlap)")

## 4. Model Inference Strategy

The top-3 most probable options from the model are used